In [4]:
%reset -f

import sys
if len(sys.argv) > 1: method = sys.argv[1]
from ROOT import gRandom, TH1, TH1D, TCanvas, cout
import ROOT

import numpy as np
import pandas as pd
from array import array
import math 



intial_path = "/w/hallb-scshelf2102/clas12/bulgakov/projects/rga_valera/analysis/"

fData = ROOT.TFile.Open("/lustre24/expphy/volatile/clas12/bulgakov/rga_valera/analysis_output/Data_xsec_MCcorr_SMfix_Jul22.root","READ")
fSim = ROOT.TFile.Open("/lustre24/expphy/volatile/clas12/bulgakov/rga_valera/analysis_output/SIM_xsec_MCcorr_SMfix_Jul22.root","READ")

def lumi(charge):
    RD=57.1;
    qe    = 1.602177E-19;    # Electron charge ---------- Coulomb;
    rho   = 0.0701;         # Density of H2 at 20K ----- g/cm3
    A0    = 6.0221367E23;    # Avogado number ----------- mol^-1
    MH    = 1.00794;         # Atomic mass of hydrogen -- amu =g/mol
    LT    = 5.0;             # Lenght of target --------- cm
    CMB   = 1e30;            # cm^2 to microbarn --------mcbarn/cm^2
    chargeConvered = charge/1e9;           # nanofarad to farad
    np           = LT*rho*A0/MH;    # number of target nuclei per cm2 (Density of target )
    ne           = chargeConvered/qe;      # Number of electron hitting the target

    factor = (ne*np)/CMB;
    return factor;

# q -bin number 0-20 
def normalize(histIN, histOUT, q, charge):
    
    deltaW = 0.05;
    e =  2.718;
    
    # bin Q2 width
    nQ2 = 50;
    q2Min = 1;
    q2Max = 2500;
    deltaQ2 = np.log(q2Max/q2Min)/nQ2;
    
    adjust = lumi(charge)/6.0;
    
    q2 =  (q2Min*math.exp( q*deltaQ2) + q2Min* math.exp((q + 1) *deltaQ2))/2;
    binQ2 = q2Min* math.exp( (q + 1) *deltaQ2) - q2Min* math.exp( q*deltaQ2);
    
    for t in range(1, 1 + histIN.GetNbinsX(),1):
        histOUT.SetBinContent(t, histIN.GetBinContent(t)/deltaW/binQ2/adjust);
        histOUT.SetBinError(t, histIN.GetBinError(t)/deltaW/binQ2/adjust);
        
def ApplyCorr(histIN, histOUT, q, df):
    
    if (q < 6 or q > 14):
        print ('no corrections for the hist out of Q2 range')
        for iXbin in range(1, 1 + histIN.GetNbinsX(),1):
            histOUT.SetBinContent(iXbin, 0.)
            histOUT.SetBinError(iXbin, 0.)
        return;
    
    nQ2 = 50;
    q2Min = 1;
    q2Max = 2500;
    deltaQ2 = np.log(q2Max/q2Min)/nQ2;
    
    # center of the bin
    q2 =  (q2Min* math.exp( q*deltaQ2) + q2Min* math.exp( (q + 1) *deltaQ2))/2
    #nW bins:
    nW = df.shape[0]/9
    
    q2binsDF = pd.unique(df['Q2'])
    df = df[df['Q2'] == q2binsDF[q-6]]
    
    if (abs(q2 - q2binsDF[q-6]) >  0.2):
        print ('Q2 range Error')
        return
    
    for iXbin in range(1, 1 + histIN.GetNbinsX(),1):

        if (histIN.GetBinCenter(iXbin)<1.14  or histIN.GetBinCenter(iXbin) > 2.5):
            continue
        corrFound = False
        for index, row in df.iterrows():
            if (abs(row['W'] - histIN.GetBinCenter(iXbin)) < 0.01):
                corrFound = True
                corr = row['Combination']
                evYield = histIN.GetBinContent(iXbin)
                histOUT.SetBinContent(iXbin, evYield * corr)
                histOUT.SetBinError(iXbin, histIN.GetBinError(iXbin) * corr)
                break
                
        if (not corrFound):
            
            if (histIN.GetBinCenter(iXbin)>2.45):
                print(row['W'], histIN.GetBinCenter(iXbin))
            histOUT.SetBinContent(iXbin, 0.)
            histOUT.SetBinError(iXbin, 0.)

def GetCorrFactorFCC(curr):
    if (curr > 0):
        return (1 - (curr - 45.)* 0.0041)
    else:
        return 1.

fcc_55 = 4.67104e+06 * GetCorrFactorFCC(55.)
fcc_50 = 3.02021e+06 * GetCorrFactorFCC(50.)
fcc_52 = 365707 * GetCorrFactorFCC(52.)
fcc_53 = 198954 * GetCorrFactorFCC(53.)
fcc_54 = 363850 * GetCorrFactorFCC(54.)

# 45nA runs
fcc_data = 5.85024e+06 * 9.8088 / 10.61360 + 6.00024e+06  + 9.37112e+06
fcc_data +=(fcc_55 + fcc_50 + fcc_52 + fcc_53 + fcc_54)
fccConst = fcc_data
print(fccConst)

29113878.952310942


In [5]:
#2D unfolding rooUnfold
def getResponse2D(fileSim, MSL, TSML, RSL, ORSL):
    for sec in range(6):
        fold = 'Deconv2D/S' + str(sec) + '/'
        endR = '_my_secBySec_S' + str(sec)
        endR2 = '_secBySec_S' + str(sec)
        
        MSL.append(fileSim.Get(fold + 'Sep_measured2D'+ endR))
        nBinsX = MSL[-1].GetNbinsX()
        TSML.append(fileSim.Get(fold + 'Sep_truth2D'+ endR))
        RSL.append(fileSim.Get(fold + 'R2D'+'_S'+str(sec)))      
        ORSL.append(ROOT.RooUnfoldResponse(MSL[-1],TSML[-1], RSL[-1]))
        
#2D to 1D
def AntiLinear2D(histIN_1, listHistsOUT, nameEnd = ''):
    nQ2bins = 11
    
    wMin_fold = histIN_1.GetXaxis().GetXmin()
    wMax_fold = histIN_1.GetXaxis().GetXmax()
    nBinsX = histIN_1.GetNbinsX()
    
    for iQ2bin in range(nQ2bins):

        histN = 'wUnf_Q2_' + str(iQ2bin + 1)
        listHistsOUT.append( ROOT.TH1D(histN + nameEnd,histN + nameEnd, nBinsX, wMin_fold, wMax_fold))
        for iBin in range(1 , 1 + nBinsX,1):
            shift = 0
            listHistsOUT[-1].SetBinContent(iBin, histIN_1.GetBinContent(iBin, iQ2bin+1))
            listHistsOUT[-1].SetBinError(iBin, histIN_1.GetBinError(iBin, iQ2bin+1))
            
            
        
# GetXSEC using 2D rooUnfold methods        
def GetXSEC_roo2D(fileSim, fileData, outXSEC, addN, chargeFC_tmp, regParam, method = 'bayes'):
    # regParam is nIter for bayes or i_comp for SVD
    if (method!= 'bayes' and method!= 'bbb' and method!= 'svd'):
        print('unsupported deconvolution method is used, please put bayes or bbb or svd')
        return
    
    #sectors
    nSec = 6
    #number of Q2 bins
    nQ2bins = 11
    
    # initialization of internal rooUnfold matrices and struct.
    tmp_measuredL = []
    tmp_truthL = []
    tmp_respMatL = []
    tmp_respObjL = []

    # read root file Sim
    getResponse2D(fileSim, tmp_measuredL, tmp_truthL, tmp_respMatL, tmp_respObjL)

    # measured from Data
    tmp_hMeas = []

    # read root file Data
    for sec in range(nSec):
        fold = 'Deconv2D/S' + str(sec) + '_my/'
        tmp_hMeas.append(fileData.Get(fold + 'measured2D_DATA_S' + str(sec)))
        

    #RooUnfoldObjs
    tmp_unfold = []
    tmp_hUnfold = []

        
    for sec in range(nSec):
        
        # method selection
        if (method == 'bayes'):
            tmp_unfold.append(ROOT.RooUnfoldBayes(tmp_respObjL[sec],tmp_hMeas[sec], regParam))
            
        if (method == 'bbb'):            
            tmp_unfold.append(ROOT.RooUnfoldBinByBin(tmp_respObjL[sec],tmp_hMeas[sec]))
        if (method == 'svd'):
            tmp_unfold.append(ROOT.RooUnfoldSvd(tmp_respObjL[sec],tmp_hMeas[sec], regParam))
            
        #####################################
        # errors accounting
        tmp_unfold[-1].IncludeSystematics(0)
        tmp_unfold[-1].SetOverflow(0)
        #tmp_unfold[-1].SetVerbose(2)
        ######################################
        
        # Actual deconvolution
        tmp_hUnfold.append(tmp_unfold[-1].Hunfold())
                   
        
    # Convert 2D hist into list of 1D hists
    wYield1D = []
    
    for sec in range(nSec): 
        wYield1D.append([])
        AntiLinear2D(tmp_hUnfold[sec], wYield1D[-1], addN + 'AL' + str(sec)) 
    
    # read file with all the corrections factors
    #corrDF = pd.read_csv('/home/valerii/Clas12/Inclusive/xsecs/rc_bcc_et_corr.dat')
    corrDF = pd.read_csv('/home/valerii/clas12/Inclusive/xsecs/rc_bcc_et_corr_targetFix.dat')

    
    
    # W yield corrected for Lumin. 
    yieldNorm = []

    for iQ2 in range(nQ2bins):

        yieldNorm.append([])
        outXSEC.append([])

        for sec in range(nSec):
            # Q2 convert binning to match binning in ana12.
            q = iQ2 + 5

            WcurrentBin = wYield1D[sec][iQ2]
            yieldNorm[-1].append(ROOT.TH1D('nrY' + str(iQ2)+ ' ' + str(sec)+ addN ,
                                           'nrY' + str(iQ2)+ ' ' + str(sec)+ addN,
                                           WcurrentBin.GetNbinsX(),
                                           WcurrentBin.GetXaxis().GetXmin(), 
                                           WcurrentBin.GetXaxis().GetXmax()))

            outXSEC[-1].append(ROOT.TH1D('xsec' + str(iQ2)+ ' ' + str(sec) + addN,
                                         'xsec' + str(iQ2)+ ' ' + str(sec)+ addN,
                                         WcurrentBin.GetNbinsX(),
                                         WcurrentBin.GetXaxis().GetXmin(), 
                                         WcurrentBin.GetXaxis().GetXmax()))

            normalize(WcurrentBin, yieldNorm[-1][-1], q, chargeFC_tmp)
            
            ApplyCorr(yieldNorm[-1][-1], outXSEC[-1][-1], q, corrDF)    




    out_file = ROOT.TFile(addN + "_unfolded_output.root", "RECREATE")

    # Save unfolded 2D hists
    out_file.mkdir("Unfold2D")
    out_file.cd("Unfold2D")
    for sec in range(nSec):
        h2 = tmp_hUnfold[sec]
        h2.SetName(f"hUnfold2D_sec{sec}")
        h2.Write()

    # Save 1D W-yield hists
    out_file.mkdir("WYield1D")
    out_file.cd("WYield1D")
    for sec in range(nSec):
        out_file.mkdir(f"WYield1D/sec{sec}")
        out_file.cd(f"WYield1D/sec{sec}")
        for iQ2 in range(nQ2bins):
            h1 = wYield1D[sec][iQ2]
            h1.SetName(f"wYield1D_sec{sec}_Q2bin{iQ2}")
            h1.Write()

    out_file.Close()
    print(f"Saved output ROOT file: {addN}_unfolded_output.root")
    
def IntOverSec(listIn, listOut, endN, isTenBin = False):
    nQ2bins = 11
    if (isTenBin):
        nQ2bins = 10
    for iQ2 in range(nQ2bins):
        listOut.append(ROOT.TH1D('intCutFunc' + str(iQ2) + endN,
                                     'intCutFunc' + str(iQ2) + endN,
                                     listIn[0][0].GetNbinsX(),
                                     listIn[0][0].GetXaxis().GetXmin(),
                                     listIn[0][0].GetXaxis().GetXmax()))
        
        for iXbin in range(1, 1 + listIn[0][0].GetNbinsX(),1):
            value = 0
            for sec in range(6):
                value += listIn[iQ2][sec].GetBinContent(iXbin)
            listOut[-1].SetBinContent(iXbin, value/6)

def GetTitle(npad):
    nQ2 = 50;
    q2Min = 1;
    q2Max = 2500;
    deltaQ2 = np.log(q2Max/q2Min)/nQ2;
    Q2bin_min = q2Min*math.exp((npad + 5)*deltaQ2)
    Q2bin_max = q2Min*math.exp( (npad + 6)*deltaQ2)
    return  "             " + str(Q2bin_min)[0:4] + " < Q^{2} < " +  str(Q2bin_max)[0:4] + " GeV^{2}"

In [6]:
epochs = [1,2,3]

xsec_allS_roo2D_bayes_1 = []
xsec_allS_roo2D_bayes_2 = []
xsec_allS_roo2D_bayes_3 = []

GetXSEC_roo2D(fSim, fData, xsec_allS_roo2D_bayes_2, 'bayes2_2D_2', fccConst, epochs[1], method = 'bayes')

xsec_integrated_2D_bayes_1 = []
xsec_integrated_2D_bayes_2 = []
xsec_integrated_2D_bayes_3 = []

IntOverSec(xsec_allS_roo2D_bayes_2, xsec_integrated_2D_bayes_2, 'xsec_integrated_2D_bayes_2')

"""
xsec_allS_roo2D_bayes_2_lastQ2 = []
GetXSEC_roo2D(fSim_lastQ2, fData_lastQ2, xsec_allS_roo2D_bayes_2_lastQ2, 'bayes2_2D_2_lastQ2', fccConst, epochs[1], method = 'bayes')
xsec_integrated_2D_bayes_2_lastQ2 = []
IntOverSec(xsec_allS_roo2D_bayes_2_lastQ2, xsec_integrated_2D_bayes_2_lastQ2, 'xsec_integrated_2D_bayes_2_lastQ2')
"""


AttributeError: Failed to get attribute RooUnfoldResponse from ROOT

Error in <TInterpreter::TCling::AutoLoad>: failure loading library libRooUnfold.so for RooUnfoldResponse
Error in <TInterpreter::TCling::AutoLoad>: failure loading library libRooUnfold.so for RooUnfoldResponse
Error in <TInterpreter::TCling::AutoLoad>: failure loading library libRooUnfold.so for RooUnfoldResponse
Error in <TInterpreter::TCling::AutoLoad>: failure loading library libRooUnfold.so for RooUnfoldResponse
Error in <TInterpreter::TCling::AutoLoad>: failure loading library libRooUnfold.so for RooUnfoldResponse
Error in <TInterpreter::TCling::AutoLoad>: failure loading library libRooUnfold.so for RooUnfoldResponse
Error in <TInterpreter::TCling::AutoLoad>: failure loading library libRooUnfold.so for RooUnfoldResponse
Error in <TInterpreter::TCling::AutoLoad>: failure loading library libRooUnfold.so for RooUnfoldResponse
Error in <TInterpreter::TCling::AutoLoad>: failure loading library libRooUnfold.so for RooUnfoldResponse
Error in <TInterpreter::TCling::AutoLoad>: failure load

In [ ]:

#statDF = pd.read_csv('/home/valerii/Clas12/Inclusive/fromLaptop/Unfolding/XSEC_final/statErrF.dat')
statDF = pd.read_csv('stat_combL.dat')

def AddSysUNC(histIN_L, df):
    
    for iQ2 in range(1,10,1):
        
        histIN = histIN_L[iQ2]
        
        # convert binning
        q = iQ2 + 5

        nQ2 = 50;
        q2Min = 1;
        q2Max = 2500;
        deltaQ2 = np.log(q2Max/q2Min)/nQ2;
        # center of the bin
        q2 =  (q2Min* math.exp( q*deltaQ2) + q2Min* math.exp( (q + 1) *deltaQ2))/2

        q2binsDF = pd.unique(df['Q2'])
        #print(q-6, q2binsDF)
        oneQ2_df = df[df['Q2'] == q2binsDF[q-6]]

        if (abs(q2 - q2binsDF[q-6]) >  0.2):
            print ('Q2 range Error')
            return

        for iXbin in range(1, 1 + histIN.GetNbinsX(),1):
            corrFound = False
            for index, row in oneQ2_df.iterrows():
                if (abs(row['W'] - histIN.GetBinCenter(iXbin)) < 0.01):
                    corrFound = True
                    #print(row['W'], histIN.GetBinCenter(iXbin))
                    corr = row['stat_combL']
                    evYield = histIN.GetBinContent(iXbin)
                    histIN.SetBinError(iXbin, evYield * corr/100)
                    break

            if (not corrFound):
                histIN.SetBinContent(iXbin, 0.)
                histIN.SetBinError(iXbin, 0.)

AddSysUNC(xsec_integrated_2D_bayes_2, statDF)
#AddSysUNC(xsec_integrated_2D_bayes_2_lastQ2, statDF)


In [ ]:
def GetSysHists(xsecIN,sysOut,df,nameH):
    
    for iQ2 in range(10):
        histD = xsecIN[iQ2]
        sysOut.append(ROOT.TH1D('intSysF' + str(iQ2) + nameH,
                                'intSysF' + str(iQ2) + nameH,
                                histD.GetNbinsX(),
                                histD.GetXaxis().GetXmin(),
                                histD.GetXaxis().GetXmax()))
        
        
        # convert binning
        q = iQ2 + 5

        nQ2 = 50;
        q2Min = 1;
        q2Max = 2500;
        deltaQ2 = np.log(q2Max/q2Min)/nQ2;
        # center of the bin
        q2 =  (q2Min* math.exp( q*deltaQ2) + q2Min* math.exp( (q + 1) *deltaQ2))/2

        q2binsDF = pd.unique(df['Q2'])
        #print(q-6, q2binsDF)
        oneQ2_df = df[df['Q2'] == q2binsDF[q-6]]

        if (abs(q2 - q2binsDF[q-6]) >  0.2):
            print ('Q2 range Error')
            continue
        
        for iXbin in range(1, 1 + histD.GetNbinsX(),1):
            corrFound = False
            for index, row in oneQ2_df.iterrows():
                if (abs(row['W'] - histD.GetBinCenter(iXbin)) < 0.01):
                    corrFound = True
                    #print(row['W'], histIN.GetBinCenter(iXbin))
                    corr = row['Total']
                    sysOut[-1].SetBinContent(iXbin, 1000 * histD.GetBinContent(iXbin)*corr/100)
                    break

            if (not corrFound):
                sysOut[-1].SetBinContent(iXbin, 0.)
                sysOut[-1].SetBinError(iXbin, 0.)


sysUncHistL = []
sysUncHistL_lastQ2 = []

sysDF = pd.read_csv('totalSysReadable_Jan2024_scale.dat')
#sysDF = pd.read_csv('/home/valerii/Clas12/Inclusive/systematics/totalSysReadable.dat')
sysDF = sysDF[['Q2','W','Total_withScale']]
sysDF = sysDF.rename(columns={"Total_withScale": "Total"})

GetSysHists(xsec_integrated_2D_bayes_2, sysUncHistL, sysDF, 'sys_Name')
#GetSysHists(xsec_integrated_2D_bayes_2_lastQ2, sysUncHistL_lastQ2, sysDF, 'sys_Name_lastQ2')

In [ ]:
!pwd

In [ ]:
#get Misak XSEC

def gerWorldData(names, result, isError):
    for name in names:
        df = pd.read_csv('/home/valerii/clas12/Inclusive/xsecs/worldData/StrFunCS/' + name, sep = '\t')
        x,y,err = array( 'd' ), array( 'd' ), array( 'd' )
        
        sp = 0
        if (name == 'Q2_3d24.dat'):
            sp = 1
        
        #isOdd = True
        for x_tmp in df.iloc[sp::2,0]:
            #if (isOdd):
                x.append(x_tmp)
                #isOdd = False
            #else:
                #isOdd = True
        
        #isOdd = True
        for y_tmp in df.iloc[sp::2,1]:
            #if (isOdd):
                y.append(1000*y_tmp)
                #isOdd = False
            #else:
                #isOdd = True
                
       #isOdd = True
        for err_tmp in df.iloc[sp::2,2]:
            #if (isOdd):
                err.append(1000*err_tmp)
                #isOdd = False
            #else:
                #isOdd = True
        if (isError):
            result.append(ROOT.TGraphErrors( len(x), x, y,ROOT.nullptr, err))
        else:
            result.append(ROOT.TGraphErrors( len(x), x, y,ROOT.nullptr, ROOT.nullptr))


def getModelCS(filePath, gr, nColumns, corr_fac):
    dfCS = pd.read_csv(filePath, sep = "\t")
    q2list = dfCS.iloc[:,0].unique()
    
    factor = 1
    if (corr_fac == 0):
        factor = 1/1000
    if (corr_fac == 1):
        factor = (1/100) * 2 / 3.1415 
    
    for q2bin in dfCS.iloc[:,0].unique():
        bin = dfCS[dfCS.iloc[:,0] == q2bin]
        x,y = array( 'd' ), array( 'd' )
        for x_tmp in bin.iloc[:,1]:
            x.append(x_tmp)
        for y_tmp in bin.iloc[:,nColumns - 1]:
            y.append(y_tmp * factor)
            
        #print(x,y)

        gr.append(ROOT.TGraph( bin.shape[0], x, y))

In [ ]:
import numpy as np
import math 

def getP(Q2,W):
    Ebeam = 10.6041
    prMass= 0.938272
    return Ebeam - (W**2 + Q2 -prMass**2) / (2 * prMass)

def getOmega(Q2,W):
    prMass= 0.938272
    return (W**2 + Q2 -prMass**2) / (2 * prMass);

def getTheta(Q2,W):
    Ebeam = 10.6041
    return 2*np.arcsin(math.sqrt(Q2/ (4* Ebeam *getP(Q2,W))))*180/3.1415926

def getThetaRAD(Q2,W):
    Ebeam = 10.6041
    return 2*np.arcsin(math.sqrt(Q2/ (4* Ebeam *getP(Q2,W))))


def setHistParam1D_3by3(inhist, color = -1, labelX = "W (GeV)"
                        , labelY = "#frac{d#sigma}{dWdQ^{2}} (#frac{nb}{GeV^{3}})"
                        , title = '', nolables = False):
    
    inhist.GetYaxis().SetNdivisions(8)
    inhist.GetXaxis().SetNdivisions(10)
    ROOT.TGaxis.SetMaxDigits(3)
    #ROOT.gPad.SetGrid()
    ROOT.gStyle.SetErrorX(0) 
    ROOT.gStyle.SetOptStat(0)
    
    inhist.SetMarkerColor(color)
    inhist.SetMarkerSize(0.8)
    inhist.SetMarkerStyle(20)
    if (title != -1):
        inhist.SetLineColor(color)
    inhist.SetLineWidth(1)
    if (title != ''):
        inhist.SetTitle(title)
    if (not nolables):
        #ROOT.TGaxis.SetMaxDigits(3)
        inhist.GetYaxis().SetTitle(labelY)
        inhist.GetXaxis().SetTitle(labelX)
    inhist.GetYaxis().SetTitleOffset(1.02)
    inhist.GetXaxis().SetTitleOffset(0.5)
    
    inhist.GetYaxis().SetTitleSize(0.09)
    inhist.GetXaxis().SetTitleSize(0.09)
    
    inhist.GetYaxis().SetLabelSize(0.09)
    inhist.GetXaxis().SetLabelSize(0.09)
    inhist.SetMinimum(0.)
    inhist.SetAxisRange(1.125,2.525,'x')
    ROOT.gStyle.SetTitleFontSize(0.2)
    # left, right, bot, up
    ROOT.gPad.SetMargin(0.195,0.02,0.14,0.17)
    
    ROOT.gStyle.SetTitleAlign(33)
    ROOT.gStyle.SetTitleX(0.9)
    ROOT.gStyle.SetTitleY(1.05)

def plotINTwithSys(listIn, listIn2, nameS, alldataPointsNP):
    
    pathCristy = 'worldData/ChristyCS.dat'
    pathMisak = 'worldData/misakModel.dat'
    pathMisak2 = 'worldData/csMis.dat'
    
    wrdGr = []
    worldData = ['Q2_2d365.dat', 'Q2_2d77.dat', 'Q2_3d24.dat', 'Q2_3d785.dat', 'Q2_4d3.dat']
    gerWorldData(worldData, wrdGr, True)
    
    """
    grMisak = []
    grMisak2 = []
    grChristy = []
    getModelCS(pathMisak, grMisak,4, 0)
    getModelCS(pathCristy, grChristy,3, 1)
    getModelCS(pathMisak2, grMisak2,3, 0)
    """  
    c2 = ROOT.TCanvas("c2","c2",1000,1300)
    c2.Divide(2,5,0.0001,0.0001)
    c2.Draw()
        
    for iQ2 in range(1,10,1):
        iPad = iQ2
        c2.cd(iQ2)
        
        hist = listIn[iQ2]
        histSys = listIn2[iQ2]
        setHistParam1D_3by3(histSys,color = 1)
        setHistParam1D_3by3(hist,color = 1)
        histSys.SetFillColor(1)
        if (iQ2 == 9):
            hist.SetAxisRange(1.125,2.29,'x')
        hist.SetTitle(GetTitle(iPad))
        hist.SetMarkerSize(0.5)
        hist.Draw('P')
        #histSys.Draw('hist same')
        
        if (iPad < 4):
            world_data2 = wrdGr[iPad]
            world_data2.SetLineColor(2);
            world_data2.SetMarkerColor(2);
            world_data2.SetLineWidth(1);
            world_data2.SetMarkerStyle(20);
            world_data2.SetMarkerSize(0.3);
            world_data2.Draw("P same")
        hist.Draw('P same')

        q = iQ2 + 5

        nQ2 = 50;
        q2Min = 1;
        q2Max = 2500;
        deltaQ2 = np.log(q2Max/q2Min)/nQ2;
        # center of the bin
        q2 =  (q2Min* math.exp( q*deltaQ2) + q2Min* math.exp( (q + 1) *deltaQ2))/2
        
        for iXbin in range(1, 1 + hist.GetNbinsX(),1):
            w_tmp = hist.GetBinCenter(iXbin)
            if (w_tmp > 1.14 and w_tmp < 2.501):
                if (q2>9 and w_tmp > 2.26):
                    continue

                k2_tmp = (q2 + getOmega(q2, w_tmp)**2)
                tan2 = (math.tan(getThetaRAD(q2, w_tmp)/2))**2 
                prMass= 0.938272
                epsil_tmp =  1 / (1 + 2* k2_tmp * tan2 /q2 ) 
                
                dataPointNP = np.array([round(q2,3), round(w_tmp,3), round(epsil_tmp,4),hist.GetBinContent(iXbin), hist.GetBinError(iXbin), histSys.GetBinContent(histSys.FindBin(w_tmp))]) 
                alldataPointsNP.append(dataPointNP)

    ROOT.gPad.GetCanvas().Print('plots/' + nameS + '.png',"png" ); 



tableNP_first = []
tableNP_last = []
plotINTwithSys(xsec_integrated_2D_bayes_2, sysUncHistL, 'xsec_sys_Jan2024_first', tableNP_first)
#plotINTwithSys(xsec_integrated_2D_bayes_2_lastQ2, sysUncHistL_lastQ2, 'xsec_sys_Jan2024_last', tableNP_last)



tableNP_stack = np.stack( tableNP_first, axis=0 )


df = pd.DataFrame(tableNP_stack, columns = ['Q2','W', 'epsilon', 'XSEC','Stat', 'Sys'])
df.to_csv("inclusive_Jan20_2024_first.dat", index=False)

#tableNP_stack_last = np.stack( tableNP_last, axis=0 )
#df = pd.DataFrame(tableNP_stack_last, columns = ['Q2','W', 'epsilon', 'XSEC','Stat', 'Sys'])
#df.to_csv("inclusive_Jan20_2024_last.dat", index=False)



In [ ]:
def plotINTwithSys_comb(listIn, listIn2, nameS, alldataPointsNP):
    
    pathCristy = 'worldData/ChristyCS.dat'
    pathMisak = 'worldData/misakModel.dat'
    pathMisak2 = 'worldData/csMis.dat'
    
    wrdGr = []
    worldData = ['Q2_2d365.dat', 'Q2_2d77.dat', 'Q2_3d24.dat', 'Q2_3d785.dat', 'Q2_4d3.dat']
    gerWorldData(worldData, wrdGr, True)
    
    """
    grMisak = []
    grMisak2 = []
    grChristy = []
    getModelCS(pathMisak, grMisak,4, 0)
    getModelCS(pathCristy, grChristy,3, 1)
    getModelCS(pathMisak2, grMisak2,3, 0)
    """  
    c2 = ROOT.TCanvas("c2","c2",1000,1300)
    #c2 = ROOT.TCanvas("c2","c2",700,1300)
    c2.Divide(2,5,0.0001,0.0001)
    #c2.Divide(1,3,0.0001,0.0001)

    #c2 = ROOT.TCanvas("c2","c2",7000,3000)
    #c2.Divide(3,3,0.0001,0.0001)
    
    c2.Draw()
        
    for iQ2 in range(1,10,1):
    #for iQ2 in range(1,4,1):
        iPad = iQ2
        c2.cd(iQ2)
        
        hist = listIn[0][iQ2]
        histSys = listIn2[0][iQ2]
        if  (iQ2 >7):
            hist = listIn[1][iQ2]
            histSys = listIn2[1][iQ2]
            
        setHistParam1D_3by3(histSys,color = 1)
        setHistParam1D_3by3(hist,color = 1)
        histSys.SetFillColor(1)
        if (iQ2 == 9):
            hist.SetAxisRange(1.125,2.29,'x')
        hist.SetTitle(GetTitle(iPad))
        hist.SetMarkerSize(0.5)
        #hist.SetMarkerSize(1.5)
        
        hist.SetMinimum(0.00000001)
        histSys.SetMinimum(0.00000001)
        #ROOT.gPad.SetLogy()
        hist.Scale(1000)
        ROOT.gPad.SetMargin(0.195,0.02,0.17,0.13)
        hist.GetXaxis().SetTitleOffset(0.91)
        hist.Draw('P')
        histSys.Draw('hist same')
        
        if (iPad < 4):
            world_data2 = wrdGr[iPad]
            world_data2.SetLineColor(2);
            world_data2.SetMarkerColor(2);
            world_data2.SetLineWidth(1);
            world_data2.SetMarkerStyle(20);
            world_data2.SetMarkerSize(0.5);
            #world_data2.SetMarkerSize(1.5);
            #world_data2.Draw("P same")
        #hist.Draw('P same')

        q = iQ2 + 5

        nQ2 = 50;
        q2Min = 1;
        q2Max = 2500;
        deltaQ2 = np.log(q2Max/q2Min)/nQ2;
        # center of the bin
        q2 =  (q2Min* math.exp( q*deltaQ2) + q2Min* math.exp( (q + 1) *deltaQ2))/2
        
        for iXbin in range(1, 1 + hist.GetNbinsX(),1):
            w_tmp = hist.GetBinCenter(iXbin)
            if (w_tmp > 1.14 and w_tmp < 2.501):
                if (q2>9 and w_tmp > 2.26):
                    continue

                k2_tmp = (q2 + getOmega(q2, w_tmp)**2)
                tan2 = (math.tan(getThetaRAD(q2, w_tmp)/2))**2 
                prMass= 0.938272
                epsil_tmp =  1 / (1 + 2* k2_tmp * tan2 /q2 ) 

                xsecOsp = 0
                xsecOspEr = 0
                if (iPad < 4):
                    for i_OSP in range(wrdGr[iPad].GetN()):
                        index = (i_OSP)
                        if (abs(w_tmp - wrdGr[iPad].GetPointX(index)) < 0.005):
                            xsecOsp = wrdGr[iPad].GetPointY(index)
                            xsecOspEr = wrdGr[iPad].GetErrorY(index)
                
                dataPointNP = np.array([round(q2,3), round(w_tmp,3), round(epsil_tmp,4),hist.GetBinContent(iXbin), hist.GetBinError(iXbin), 
                                        histSys.GetBinContent(histSys.FindBin(w_tmp)), xsecOsp, xsecOspEr]) 
                alldataPointsNP.append(dataPointNP)

    #ROOT.gPad.GetCanvas().Print('plots/' + nameS + '_3bins.png',"png" ); 
    ROOT.gPad.GetCanvas().Print('plots/' + nameS + '_noOSP.png',"png" ); 

tableNP_comb = []
plotINTwithSys_comb([xsec_integrated_2D_bayes_2,xsec_integrated_2D_bayes_2_lastQ2], 
                    #[sysUncHistL,sysUncHistL_lastQ2], 'xsec_sys_Jan2024_comb_ftof', tableNP_comb)
                    [sysUncHistL,sysUncHistL_lastQ2], 'xsec_sys_Jan2024_comb', tableNP_comb)

tableNP_stack_comb = np.stack( tableNP_comb, axis=0 )


df = pd.DataFrame(tableNP_stack_comb, columns = ['Q2','W', 'epsilon', 'XSEC','Stat', 'Sys','Osp', 'OspErr'])
df.to_csv("inclusive_Jan20_2024_comb_totalUpd_Osp.dat", index=False)

In [ ]:
dXSEC_comb = pd.read_csv('inclusive_Jan20_2024_comb_totalUpd.dat') # no Q2_N column
dXSEC_comb['ScaleType'] = dXSEC_comb.apply(lambda row: math.sqrt(0.5**2 + 0.5**2 + 1.0**2 + 1.0**2 + 3.0**2)*row.XSEC/100
                                           , axis=1)

#pd.set_option('display.float_format', '{:.3E}'.format)
dXSEC_comb.to_csv("inclusive_Jan20_2024_comb_full_2_totalUpd.dat", index=False)
dXSEC_comb

In [ ]:
import matplotlib.pyplot as plt




df = df[df.Q2 == 9.699]
    
plt.figure()
df.plot(x = 'W', y = 'XSEC', label='',
        color = 'mediumseagreen', grid = True, kind ='area')
    
df.plot(x = 'W', y = 'Sys', label='',
        color = 'blue', grid = True, kind ='area')

title = " $\sigma$ dp/p inclusive"
    
plt.title(title, fontsize=30) 
plt.ylabel("$\sigma$", fontsize=30)
plt.xlabel('$W$, [GeV]', fontsize=30)
#plt.tick_params(axis='both', which='major', labelsize=40)
#axes.set_ylim([0, 25])


#plt.legend(fontsize=10)

plt.savefig('plots/PANDAStest.png', bbox_inches='tight', dpi=300)

In [ ]:
def plotSECs(listIn, nameS):
        
    c2 = ROOT.TCanvas("c2","c2",3500,1500)
    c2.Divide(3,3,0.0001,0.0001)
    c2.Draw()
        
    for iQ2 in range(1,10,1):
        iPad = iQ2
        c2.cd(iQ2)
        colorSec = [858,858,807,1,807,858]
        for sec in range(6):
            hist = listIn[iQ2][sec]
            setHistParam1D_3by3(hist,color = colorSec[sec])
            hist.SetMarkerSize(0.3)
            if (sec == 0):
                if (iQ2 == 9):
                    hist.SetAxisRange(1.125,2.27,'x')
                hist.SetTitle(GetTitle(iPad))
                hist.Draw('p')
            else:
                hist.Draw('p same')

    ROOT.gPad.GetCanvas().Print( nameS + '.png',"png" ); 
#858 - blue, 797 - orange    

plotSECs(xsec_allS_roo2D_bayes_2,'allS')

In [ ]:
def getOmega(Q2,W):
    prMass= 0.938272
    return (W**2 + Q2 -prMass**2) / (2 * prMass);

def getThetaRAD(Q2,W):
    Ebeam = 10.6041
    return 2*np.arcsin(math.sqrt(Q2/ (4* Ebeam *getP(Q2,W))))

k2_tmp = (q2 + getOmega(q2, w_tmp)**2)
tan2 = (math.tan(getThetaRAD(q2, w_tmp)/2))**2 
prMass= 0.938272
epsil_tmp =  1 / (1 + 2* k2_tmp * tan2 /q2 ) 

In [ ]:
oldCorrDF = pd.read_csv('rc_bcc_et_corr.dat')
oldCorrDF

In [ ]:
updCorrDF = pd.read_csv('rcANDbcc_targetFix.dat')
updCorrDF['ET'] = oldCorrDF.ET
updCorrDF['Combination'] = updCorrDF.apply(lambda row: (row.ET/ (row.BCC_targetFix_reversed * row.RC_targetFix_reversed)), axis=1)
updCorrDF = updCorrDF.rename(columns={"BCC_targetFix_reversed": "BCC_iter3_reversed", "RC_targetFix_reversed": "RC_iter3_reversed"})
updCorrDF.to_csv('rc_bcc_et_corr_targetFix' + ".dat", index=False)

In [ ]:
(5.7-36.06)/2760

In [ ]:
2760* 0.011

In [ ]:
(1.41-10.1)/0.002

In [ ]:
0.003*2870

In [ ]:
0.0025*3444

In [ ]:
(5.7-36.06)/0.005